# Train a worm to crawl
In this notebook you will get started with reinforcement learning. You will train a worm to learn to crawl.
The notebook consists of the following steps:1. You investigate how a worm crawls.2. You represent the worm abstractly as a sequence of segments.3. You define what **the agent**, **the environment**, **the state**, **the reward**, and **the actions** are.4. You look for an optimal **policy** by learning a **value function** based on the **rewards**.


## How does a worm crawl?
You've probably come across a worm while digging in the garden. Have you ever stopped to think about how such a worm moves underground? That's not so obvious, since worms don't have legs. Imagine we wrap you in duct tape and bury you in the ground—would you still be able to move?
### The morphology of a worm
We call the **structure and shape** of an organism the **morphology** of that organism.
This morphology enables the worm to move underground.
**Segments**
A worm is made up of separate segments that are filled with fluid.
![Segments of the worm](img/worm_base.png)
**Muscles**
Longitudinal muscles run along the length of the worm.
![Longitudinal muscles of the worm](img/worm_long_muscles.png)
Besides the longitudinal muscles, the worm also has circular muscles that run around the worm’s body.
![Circular muscles of the worm](img/worm_short_muscles.png)

**Little hairs**
There are also small hairs on the worm’s body. These help the worm get more grip on the surrounding soil.

### The movements of the worm
The worm can move each segment in two possible ways.
**Contracting longitudinal muscles**
By contracting the longitudinal muscles in a segment, the length of the segment becomes shorter. Because the segment is filled with fluid and fluid is incompressible, this causes the outside of the worm to expand due to the pressure.
![Shortening a segment.](img/worm_contracted.png)
Conversely, by contracting the circular muscles in a segment, the circumference of the segment becomes shorter. As a result, the segment will become longer due to the pressure.
![Expanding a segment](img/worm_expanded_segment.png)
A shortened segment increases the pressure on the soil around the worm. The bristles on the worm therefore grip the soil at that spot. An expanded segment makes the worm longer and thinner, so there is less friction on the soil around the worm and that section can move forward and backward more easily.
By contracting and relaxing the longitudinal and circular muscles in a correct pattern, the worm can move forward. It is this pattern that we will try to teach a virtual worm using reinforcement learning.

# The virtual worm
Before creating the virtual worm, we first import the necessary libraries. As always, we have already written some starter code. This allows us to focus in this notebook on what is important for training the worm. If you’d like to look at that code, you can find it in the file *scripts/helpers.py*.

In [1]:
from scripts.helpers import Worm, print_q_tabel, voer_policy_uit, maak_animatie_van_worm
from itertools import product
from scripts.questions import Questions
from IPython.display import HTML, Video
import numpy as np
import random
vragen = Questions()

To simplify the problem a bit, we represent our worm as a list of segments. Each segment can be either long or short. We omit the neutral state of the muscles, so it will be easier to learn a movement pattern.

For the virtual worm, we have created a class. It contains the list of segments and functions to make a segment long or short. The code below creates a worm with 5 segments.

In [ ]:
worm = Worm(5)

When we print the worm, we get a list of letters. Each letter represents the state of a segment, K for short and L for long. We have chosen to make all segments short when creating a worm.

In [ ]:
print(f"De starttoestand van de worm is: {worm}")

The worm object has a number of functions (methods) that we can use to change the state of the worm. The following code prints those methods.

In [ ]:
# Druk de af welke methodes de worm heeft
print("De worm heeft de volgende methodes:")
methodes = [method_naam for method_naam in dir(worm) if callable(getattr(worm, method_naam))]
for method in methodes:
    if not method.startswith("__"):
        print(f"- {method}")

You notice that there is a method to draw the worm. Let's give it a try.

In [ ]:
worm.teken_worm()

You can see that all elements of the worm are currently short. With the method `maak_segment_langer()` you can make one of the segments long. You pass the number of the segment to the method. The segments are numbered starting at 0.

In [ ]:
worm.maak_segment_langer(4)
print(f"De worm is nu: {worm}")
worm.teken_worm()

**Note**
The short segments are thick and press against the soil around the worm. The worm’s bristles grip the soil in this way. As a result, it is not possible to lengthen a segment that sits between two short segments. In that case, the worm would have to move to the left or right. But because there are short segments on both sides of the segment, that isn’t possible. For example, you can see that lengthening segment 2 has no effect.

In [ ]:
worm.maak_segment_langer(4)
print(f"De worm is nu: {worm}")
worm.teken_worm()

By shortening or lengthening segments in the right pattern, the worm can propel itself.
**Assignment**: apply a sequence of moves to the worm so that the worm's tail moves two positions to the right. After each shift, print the worm to the screen.

In [ ]:
# Schrijf hier de code om de worm te doen bewegen.


## Teaching the worm to crawl
Earlier, you devised a series of movements that could make the worm move forward. Now we want to teach the worm to move by itself using reinforcement learning. To do that, we first consider how we can link the principles of reinforcement learning to the worm. Below, we list the different principles once again.
* **The agent** is the one who will learn the task.* **The environment** is the world in which the agent will learn.* The environment also has a **state**, which is what the world looks like at that moment.* **The actions**: By performing actions, the agent can influence the state of the environment.* **A reward**: Good actions receive a reward, bad actions are punished with a negative reward. In humans and animals, a reward corresponds to the feelings of pleasure and pain.* **The policy or the policy**: This is the strategy the agent is applying at that moment. The policy specifies which action the agent will take in a given state of the environment.* **The value function**: While the reward gives you an idea of which actions were good or bad in a given state, the value function indicates what is good over the long term.
**Assignment**: Think about how you can connect these principles to the problem of teaching the worm to crawl. Run the code cell below and answer the question.

In [ ]:
vragen.stel_vraag(0)

You are now able to determine what the agent, the environment, the state, the actions, and the reward are. In what follows we want to learn a **policy** that the worm can follow to crawl. **The policy says in each state which action the worm should take** (e.g., make segment 2 longer). The worm will make this decision based on the future reward that the action will yield. **The estimate of this future reward is provided by the value function.**
It is this value function that for every combination of **state** and **action** will say how good that action is in that state. During the training of our 'reinforcement learning' system, we seek this **value function**.

## The value function
As stated, the value function will, for each combination of state and action, indicate how good that action is in that state. Mathematically, we can therefore represent the function as a function of two variables.
\begin{equation}Q(a, s)\end{equation}
Where $a$ denotes the action and $s$ the state (state in English).
In our simple system, we will represent the value function as a table. Each row of the table corresponds to a state and each column to an action. Before we can construct the table, we therefore need to know how many states and actions there are. Let's do that for a worm with 3 segments.
Our simplified worm consists of 3 segments, each segment can be either long or short. Run the following cells and answer the questions.

In [ ]:
vragen.stel_vraag(1)

In [ ]:
vragen.stel_vraag(2)

### The Q-table
The table we use to represent the value function is called the Q-table. Each element of the table contains an estimate of the future reward for a combination of an action and a state. In the code cells below, we set up that table.
First we create a list of all possible states and all possible actions.

**The states**

In [ ]:
# De toestand van de worm stellen we voor door een opeenvolging van segmenten.
# Elk segment kan ofwel kort zijn (K) ofwel lang zijn (L).
# De product functie zal alle mogelijke combinaties van segmenten genereren.
toestanden = list(product("KL", repeat=3))
print(toestanden)

**The actions**

In [ ]:
# Elk segment heeft twee mogelijke acties: langer worden of korter worden.
# We maken een lijst die voor elk segment een langer en korter actie bevat.
acties = []
for i in range(3):
    acties.append(("L", i))
    acties.append(("K", i))
    
print(acties)

**The Q-table**
The Q-table has as many rows as there are states and as many columns as there are actions. When we start, we have no information about the future reward. Therefore, we choose random values for the elements in our table.

In [ ]:
# Maak een Q-tabel waar alle elementen willekeurig worden ingevuld met een waarde tussen -1 en 1.
q_tabel = np.random.uniform(-1, 1, (len(toestanden), len(acties)))
print("Q-tabel:")
print(q_tabel)

In *helpers.py* we wrote a function to display the Q-table clearly.

In [ ]:
print_q_tabel(q_tabel, toestanden, acties)

This is the table that we will modify later during the training process.

## Execute a policy
You might be wondering how a table can represent a certain behavior. That can indeed be difficult to understand. To give you an idea of how that works, below we provide an example of a Q-table that contains a learned policy.

To do that, we load an existing Q-table from a file:

In [ ]:
voorbeeld_q_tabel = np.load("bestanden/q_tabel_korte_worm.npy")
print("Voorbeeld Q-tabel:")
print_q_tabel(voorbeeld_q_tabel, toestanden, acties)

Alright, you can see that the Q-table contains numbers. How can we now have the worm carry out the learned behavior?

To perform a behavior, first create a new worm and repeat the following commands for a certain number of steps (e.g., 20 steps).
1. Check which state the worm is currently in.2. Find the row in the Q-table that corresponds to this state.3. Find the column with the highest value in this row.4. Perform the action that corresponds to this column.

Below we provide step-by-step code to execute the policy in the Q-table.

#### Create a new worm with 3 segments.

In [ ]:
# Maak een nieuwe worm met 3 segmenten.
nieuwe_worm = Worm(3)

#### 1. Retrieve the current state of the worm.

In [ ]:
# Vraag de toestand van de worm op.
toestand_worm = nieuwe_worm.toestand()
print(f"De toestand van de worm is: {toestand_worm}")

#### 2. Find the row in the Q-table that corresponds to this state.

In [ ]:
# Zoek de index van de toestand in de lijst van toestanden.
toestand_index_worm = toestanden.index(toestand_worm)
print(f"De index van de toestand in de lijst van toestanden is: {toestand_index_worm}")

# Haal op basis van de index de rij uit de Q-tabel.
rij_q_tabel = voorbeeld_q_tabel[toestand_index_worm]
print(f"De rij in de Q-tabel die overeenkomt met de toestand van de worm is: {rij_q_tabel}")

#### 3. Find in this row the action with the highest Q value.
We can find that action based on the index of the highest value. We can retrieve this index with the *argmax()* function.

In [ ]:
# Zoek de index van de actie met de hoogste Q-waarde in die rij.
actie_index = np.argmax(rij_q_tabel)
print(f"De index van de beste actie is: {actie_index}")

# Haal de actie op basis van de index.
beste_actie = acties[actie_index]
print(f"De beste actie is: {beste_actie}")

#### 4. Execute the action

In [ ]:
# Voer de actie uit op de worm.
if beste_actie[0] == "L":   # Langer worden
    nieuwe_worm.maak_segment_langer(beste_actie[1])
else:                       # Korter worden
    nieuwe_worm.maak_segment_korter(beste_actie[1])
    
# Druk de toestand van de worm na het uitvoeren van de actie.
print(f"De toestand van de worm na het uitvoeren van de actie is: {nieuwe_worm}")

#### Repeat everything
We’ve now run all the code to make the worm take an action based on the Q-table. We will now keep repeating these steps. To do that, we can put the code in a loop. At each step we also save the worm’s state. This way we can visualize the crawling pattern. We’ll put all this code into a function here. We can then reuse this function later to visualize the policy you teach the worm.

In [ ]:
def voer_beleid_uit(voorbeeld_q_tabel, toestanden, acties, aantal_stappen=20, lengte_worm=3, bestandsnaam="animatie_worm.mp4"):
    """
    Voert het beleid uit dat is opgeslagen in de Q-tabel en maakt een animatie van de worm.
    :param aantal_stappen: Het aantal stappen dat de worm zal nemen.
    :param lengte_worm: De lengte van de worm waarmee het beleid wordt uitgevoerd.
    """
    stadia_van_de_worm = [] # Lijst waarin we elke tussenstaop opslaan.
    nieuwe_worm = Worm(lengte_worm)    # We beginnen met een nieuwe worm.
    stadia_van_de_worm.append(nieuwe_worm.maak_kopie())  # Voeg de starttoestand van de worm toe aan de lijst.

    # Hier overlopen we de stappen en voeren we de commando's uit die we hierboven stap voor stap hebben bepaald.
    for stap in range(aantal_stappen):
        toestand_worm = nieuwe_worm.toestand()                  # Vraag de toestand van de worm op.
        toestand_index_worm = toestanden.index(toestand_worm)   # Zoek de index van de toestand in de lijst van toestanden.
        rij_q_tabel = voorbeeld_q_tabel[toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
        actie_index = np.argmax(rij_q_tabel)                    # Zoek de index van de actie met de hoogste Q-waarde in die rij.    
        beste_actie = acties[actie_index]                      # Haal de actie op basis van de index. 
        # Voer de actie uit
        if beste_actie[0] == "L":
            nieuwe_worm.maak_segment_langer(beste_actie[1])
        else:
            nieuwe_worm.maak_segment_korter(beste_actie[1])
        
        # Voeg een kopie van de huidige toestand van de worm toe aan de lijst van stadia.
        stadia_van_de_worm.append(nieuwe_worm.maak_kopie())

    # Maak een animatie van de worm.
    maak_animatie_van_worm(stadia_van_de_worm, bestandsnaam=bestandsnaam)
    
bestandsnaam = "animatie_korte_worm.mp4"
voer_beleid_uit(voorbeeld_q_tabel, toestanden, acties, aantal_stappen=20, lengte_worm=3, bestandsnaam=bestandsnaam)
Video(bestandsnaam)



## Learning the optimal policy
Above, you have seen how to execute a previously learned policy. In the next step, you will have the worm learn a policy on its own. The steps required to learn a policy are similar to those for executing a policy. However, we add a number of elements. The new code in the function is shown each time between the arrows.

Below you can see the code we used above to run a policy. We have renamed the function from `voer_beleid_uit` to `leer_beleid`. The rest of the code is the same for now. We will add the necessary logic to this function step by step to enable learning.

In [ ]:
def leer_beleid(voorbeeld_q_tabel, toestanden, acties, aantal_stappen=20, lengte_worm=3):
    stadia_van_de_worm = [] # Lijst waarin we elke tussenstaop opslaan.
    nieuwe_worm = Worm(lengte_worm)    # We beginnen met een nieuwe worm.
    stadia_van_de_worm.append(nieuwe_worm.maak_kopie())  # Voeg de starttoestand van de worm toe aan de lijst.

    # Hier overlopen we de stappen en voeren we de commando's uit die we hierboven stap voor stap hebben bepaald.
    for stap in range(aantal_stappen):
        toestand_worm = nieuwe_worm.toestand()                  # Vraag de toestand van de worm op.
        toestand_index_worm = toestanden.index(toestand_worm)   # Zoek de index van de toestand in de lijst van toestanden.
        rij_q_tabel = voorbeeld_q_tabel[toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
        actie_index = np.argmax(rij_q_tabel)                    # Zoek de index van de actie met de hoogste Q-waarde in die rij.    
        beste_actie = acties[actie_index]                      # Haal de actie op basis van de index. 
        # Voer de actie uit
        if beste_actie[0] == "L":
            nieuwe_worm.maak_segment_langer(beste_actie[1])
        else:
            nieuwe_worm.maak_segment_korter(beste_actie[1])
        
        # Voeg een kopie van de huidige toestand van de worm toe aan de lijst van stadia.
        stadia_van_de_worm.append(nieuwe_worm.maak_kopie())

#### Add exploration
Now we choose to always have the worm take the best action in every state. For a previously learned policy this is fine because we know that it works well. But when the policy is not yet good enough, it is useful to explore other possibilities.
We can add this exploration by not always choosing the best action but, for example, by choosing a random action 1/10 of the time. We will do this by generating a random number between 0 and 1 and checking whether it is smaller than a given value. We call this value epsilon ($\epsilon$). The higher $\epsilon$, the more likely it is that we choose a random action. We add this logic to the code below.

In [ ]:
# We voegen epsilon toe als parameter van de functie.
def leer_beleid(voorbeeld_q_tabel, toestanden, acties, epsilon=0.1, aantal_stappen=20, lengte_worm=3):
    stadia_van_de_worm = [] # Lijst waarin we elke tussenstaop opslaan.
    nieuwe_worm = Worm(lengte_worm)    # We beginnen met een nieuwe worm.
    stadia_van_de_worm.append(nieuwe_worm.maak_kopie())  # Voeg de starttoestand van de worm toe aan de lijst.

    # Hier overlopen we de stappen en voeren we de commando's uit die we hierboven stap voor stap hebben bepaald.
    for stap in range(aantal_stappen):
        toestand_worm = nieuwe_worm.toestand()                  # Vraag de toestand van de worm op.
        toestand_index_worm = toestanden.index(toestand_worm)   # Zoek de index van de toestand in de lijst van toestanden.
        rij_q_tabel = voorbeeld_q_tabel[toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
        
        # -----------------------------------↓↓↓↓↓↓↓↓↓↓↓↓↓↓----------------------------------- #
        willekeurig_getal = np.random.rand()  # Genereer een willekeurig getal tussen 0 en 1.
        if willekeurig_getal < epsilon:
            # Kies een willekeurige actie (exploratie).
            actie_index = random.randint(0, len(acties) - 1)
        else:
            # Kies de beste actie op basis van de Q-tabel (exploitatie).
            actie_index = np.argmax(rij_q_tabel)                    # Zoek de index van de actie met de hoogste Q-waarde in die rij.  
              
        beste_actie = acties[actie_index]                      # Haal de actie op basis van de index. 
        # -----------------------------------↑↑↑↑↑↑↑↑↑↑↑↑↑↑----------------------------------- #
        
        # Voer de actie uit
        if beste_actie[0] == "L":
            nieuwe_worm.maak_segment_langer(beste_actie[1])
        else:
            nieuwe_worm.maak_segment_korter(beste_actie[1])
        
        # Voeg een kopie van de huidige toestand van de worm toe aan de lijst van stadia.
        stadia_van_de_worm.append(nieuwe_worm.maak_kopie())

#### Determine the reward of the action
In order to learn, we must be able to assess how good an action is. We do that based on the reward received. Here we choose as the reward the distance that the worm's head has traveled. In the cell below, we add the necessary code to determine the reward.

In [ ]:
# We voegen epsilon toe als parameter van de functie.
def leer_beleid(voorbeeld_q_tabel, toestanden, acties, epsilon=0.1, aantal_stappen=20, lengte_worm=3):
    stadia_van_de_worm = [] # Lijst waarin we elke tussenstaop opslaan.
    nieuwe_worm = Worm(lengte_worm)    # We beginnen met een nieuwe worm.
    stadia_van_de_worm.append(nieuwe_worm.maak_kopie())  # Voeg de starttoestand van de worm toe aan de lijst.

    # Hier overlopen we de stappen en voeren we de commando's uit die we hierboven stap voor stap hebben bepaald.
    for stap in range(aantal_stappen):
        toestand_worm = nieuwe_worm.toestand()                  # Vraag de toestand van de worm op.
        toestand_index_worm = toestanden.index(toestand_worm)   # Zoek de index van de toestand in de lijst van toestanden.
        rij_q_tabel = voorbeeld_q_tabel[toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
        
        
        willekeurig_getal = np.random.rand()  # Genereer een willekeurig getal tussen 0 en 1.
        if willekeurig_getal < epsilon:
            # Kies een willekeurige actie (exploratie).
            actie_index = random.randint(0, len(acties) - 1)
        else:
            # Kies de beste actie op basis van de Q-tabel (exploitatie).
            actie_index = np.argmax(rij_q_tabel)                    # Zoek de index van de actie met de hoogste Q-waarde in die rij.  
              
        beste_actie = acties[actie_index]                      # Haal de actie op basis van de index. 
        
        
        # -----------------------------------↓↓↓↓↓↓↓↓↓↓↓↓↓↓----------------------------------- #
        # Sla de positie van het hoofd op voor we de actie uitvoeren.
        positie_hoofd = nieuwe_worm.positie_van_het_hoofd()
        # -----------------------------------↑↑↑↑↑↑↑↑↑↑↑↑↑↑----------------------------------- #
        
        # Voer de actie uit
        if beste_actie[0] == "L":
            nieuwe_worm.maak_segment_langer(beste_actie[1])
        else:
            nieuwe_worm.maak_segment_korter(beste_actie[1])
            
        # -----------------------------------↓↓↓↓↓↓↓↓↓↓↓↓↓↓----------------------------------- #
        # De beloning is het verschil in afstand tussen de nieuwe positie van het hoofd en de oude positie.
        beloning = nieuwe_worm.positie_van_het_hoofd() - positie_hoofd
        # -----------------------------------↑↑↑↑↑↑↑↑↑↑↑↑↑↑----------------------------------- #
        
        # Voeg een kopie van de huidige toestand van de worm toe aan de lijst van stadia.
        stadia_van_de_worm.append(nieuwe_worm.maak_kopie())

#### Adjusting the Q-table using the update rule.
Next, we need to adjust the Q-table. For that, we use the update rule. You already saw the formula for this rule earlier in the learning path. Below, we repeat the formula once more.
![](img/update_function_explained.png)

Several elements from the formula are already in our code. We have the Q-table and the current state and action. We also know the reward for performing the action. However, there are still some elements from the formula that are missing.
- $\alpha$: This is a number that indicates how quickly the Q-table should be adjusted based on the reward. We can choose this number ourselves, so we add it as an argument to the function.- $\gamma$: This is a number that indicates how important the future reward is after performing an action. We can choose this value ourselves, so we also add $\gamma$ as an argument to the function.- $\max_a Q(T_{t+1}, a)$: This is the Q-value of the best action in the new state. It is therefore an estimate of the reward we can expect in this new state.

In [ ]:
# We voegen epsilon toe als parameter van de functie.
def leer_beleid(voorbeeld_q_tabel, toestanden, acties, epsilon=0.1, alpha=0.1, gamma=0.9, aantal_stappen=20, lengte_worm=3):
    stadia_van_de_worm = [] # Lijst waarin we elke tussenstaop opslaan.
    nieuwe_worm = Worm(lengte_worm)    # We beginnen met een nieuwe worm.
    stadia_van_de_worm.append(nieuwe_worm.maak_kopie())  # Voeg de starttoestand van de worm toe aan de lijst.

    # Hier overlopen we de stappen en voeren we de commando's uit die we hierboven stap voor stap hebben bepaald.
    for stap in range(aantal_stappen):
        toestand_worm = nieuwe_worm.toestand()                  # Vraag de toestand van de worm op.
        toestand_index_worm = toestanden.index(toestand_worm)   # Zoek de index van de toestand in de lijst van toestanden.
        rij_q_tabel = voorbeeld_q_tabel[toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
        
        
        willekeurig_getal = np.random.rand()  # Genereer een willekeurig getal tussen 0 en 1.
        if willekeurig_getal < epsilon:
            # Kies een willekeurige actie (exploratie).
            actie_index = random.randint(0, len(acties) - 1)
        else:
            # Kies de beste actie op basis van de Q-tabel (exploitatie).
            actie_index = np.argmax(rij_q_tabel)                    # Zoek de index van de actie met de hoogste Q-waarde in die rij.  
              
        beste_actie = acties[actie_index]                      # Haal de actie op basis van de index. 
        
        # Sla de positie van het hoofd op voor we de actie uitvoeren.
        positie_hoofd = nieuwe_worm.positie_van_het_hoofd()
        
        # Voer de actie uit
        if beste_actie[0] == "L":
            nieuwe_worm.maak_segment_langer(beste_actie[1])
        else:
            nieuwe_worm.maak_segment_korter(beste_actie[1])
            
        # De beloning is het verschil in afstand tussen de nieuwe positie van het hoofd en de oude positie.
        beloning = nieuwe_worm.positie_van_het_hoofd() - positie_hoofd
        
        
        # -----------------------------------↓↓↓↓↓↓↓↓↓↓↓↓↓↓----------------------------------- #
        
        # Zoek de beste waarde in de nieuwe toestand.
        nieuwe_toestand_worm = nieuwe_worm.toestand()                  # Vraag de nieuwe toestand van de worm op, na het uitvoeren van de actie.
        nieuwe_toestand_index_worm = toestanden.index(nieuwe_toestand_worm)   # Zoek de index van de nieuwe toestand in de lijst van toestanden.
        nieuwe_rij_q_tabel = voorbeeld_q_tabel[nieuwe_toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
        waarde_beste_actie_nieuwe_toestand = np.max(nieuwe_rij_q_tabel)       # Zoek de beste waarde in de nieuwe toestand.
        
        # Update de Q-waarde voor de huidige toestand en actie.
        q_tabel[toestand_index_worm, actie_index] += alpha * (beloning + gamma * waarde_beste_actie_nieuwe_toestand - q_tabel[toestand_index_worm, actie_index])
        
        # -----------------------------------↑↑↑↑↑↑↑↑↑↑↑↑↑↑----------------------------------- #
        
        # Voeg een kopie van de huidige toestand van de worm toe aan de lijst van stadia.
        stadia_van_de_worm.append(nieuwe_worm.maak_kopie())
        
    return stadia_van_de_worm  # Geef de lijst van stadia terug voor animatie of verdere verwerking.
        

#### Repeating the learning process
Currently, we let the worm learn by performing `aantal_stappen` actions. When the worm has found a way to move forward in a particular state, it becomes very difficult to learn that from other states as well. Therefore, we choose to repeat this learning process a number of times. At each repetition, we reset the worm to its initial state. We call each repetition an **episode**.

In [ ]:
# We voegen de parameter eppisodes toe aan de functie om het aantal episodes te bepalen.
def leer_beleid(voorbeeld_q_tabel, toestanden, acties, epsilon=0.1, alpha=0.1, gamma=0.9, aantal_stappen=50, episodes=20, lengte_worm=3):
    stadia_van_de_worm = [] # Lijst waarin we elke tussenstaop opslaan.
    
    # -----------------------------------↓↓↓↓↓↓↓↓↓↓↓↓↓↓----------------------------------- #
    for episode in range(episodes):  # Loop over het aantal episodes.
    # -----------------------------------↑↑↑↑↑↑↑↑↑↑↑↑↑↑----------------------------------- #
        
        nieuwe_worm = Worm(lengte_worm)    # We beginnen met een nieuwe worm.
        stadia_van_de_worm.append(nieuwe_worm.maak_kopie())  # Voeg de starttoestand van de worm toe aan de lijst.

        # Hier overlopen we de stappen en voeren we de commando's uit die we hierboven stap voor stap hebben bepaald.
        for stap in range(aantal_stappen):
            toestand_worm = nieuwe_worm.toestand()                  # Vraag de toestand van de worm op.
            toestand_index_worm = toestanden.index(toestand_worm)   # Zoek de index van de toestand in de lijst van toestanden.
            rij_q_tabel = voorbeeld_q_tabel[toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
            
            
            willekeurig_getal = np.random.rand()  # Genereer een willekeurig getal tussen 0 en 1.
            if willekeurig_getal < epsilon:
                # Kies een willekeurige actie (exploratie).
                actie_index = random.randint(0, len(acties) - 1)
            else:
                # Kies de beste actie op basis van de Q-tabel (exploitatie).
                actie_index = np.argmax(rij_q_tabel)                    # Zoek de index van de actie met de hoogste Q-waarde in die rij.  
                
            beste_actie = acties[actie_index]                      # Haal de actie op basis van de index. 
            
            # Sla de positie van het hoofd op voor we de actie uitvoeren.
            positie_hoofd = nieuwe_worm.positie_van_het_hoofd()
            
            # Voer de actie uit
            if beste_actie[0] == "L":
                nieuwe_worm.maak_segment_langer(beste_actie[1])
            else:
                nieuwe_worm.maak_segment_korter(beste_actie[1])
                
            # De beloning is het verschil in afstand tussen de nieuwe positie van het hoofd en de oude positie.
            beloning = nieuwe_worm.positie_van_het_hoofd() - positie_hoofd
    
            # Zoek de beste waarde in de nieuwe toestand.
            nieuwe_toestand_worm = nieuwe_worm.toestand()                  # Vraag de nieuwe toestand van de worm op, na het uitvoeren van de actie.
            nieuwe_toestand_index_worm = toestanden.index(nieuwe_toestand_worm)   # Zoek de index van de nieuwe toestand in de lijst van toestanden.
            nieuwe_rij_q_tabel = voorbeeld_q_tabel[nieuwe_toestand_index_worm]    # Haal de rij uit de Q-tabel op basis van de index.
            waarde_beste_actie_nieuwe_toestand = np.max(nieuwe_rij_q_tabel)       # Zoek de beste waarde in de nieuwe toestand.
            
            # Update de Q-waarde voor de huidige toestand en actie.
            q_tabel[toestand_index_worm, actie_index] += alpha * (beloning + gamma * waarde_beste_actie_nieuwe_toestand - q_tabel[toestand_index_worm, actie_index])
            
            # Voeg een kopie van de huidige toestand van de worm toe aan de lijst van stadia.
            stadia_van_de_worm.append(nieuwe_worm.maak_kopie())
            
    return stadia_van_de_worm  # Geef de lijst van stadia terug voor animatie of verdere verwerking.
        

#### Letting the worm learn
So, you now have a function that can let the worm learn based on the reward. Let's run this function with 100 steps and 5 episodes. Make sure you have executed the code cell above!

In [ ]:
q_tabel = np.random.uniform(-1, 1, (len(toestanden), len(acties)))
stadia = leer_beleid(q_tabel, 
                     toestanden, 
                     acties, 
                     epsilon=0.1, 
                     alpha=0.1, 
                     gamma=0.9, 
                     aantal_stappen=100, 
                     episodes=5,
                     lengte_worm=3)
bestandsnaam = "animatie_leerproces_worm.mp4"
maak_animatie_van_worm(stadia, bestandsnaam=bestandsnaam, fps=10)
Video(bestandsnaam)

Look closely at the animation! What does the worm do during learning. Do you see when the worm explores a new action and when it exploits specific knowledge?
**Assignment**: Adjust the parameters of the learning process and try to ensure that the worm crawls in an efficient way. Adjust only one parameter at a time and observe the effect.**Tips**:- Add the `fps` parameter to the `maak_animatie_van_worm()` function. This allows you to specify how many frames per second should be played. Increasing this value makes the animation faster.- You can give each animation a different name; that way you can view them again later and compare them.- Note: When you choose a high number of episodes and steps, it may take a long time to run the code.

In [ ]:
q_tabel = np.random.uniform(-1, 1, (len(toestanden), len(acties)))
stadia = leer_beleid(q_tabel, 
                     toestanden, 
                     acties, 
                     epsilon=0.1, 
                     alpha=0.1, 
                     gamma=0.9, 
                     aantal_stappen=100, 
                     episodes=5,
                     lengte_worm=3)

bestandsnaam = "animatie_leerproces_worm.mp4"
maak_animatie_van_worm(stadia, bestandsnaam=bestandsnaam, fps=10)
Video(bestandsnaam)

#### Execute the learned policy
If you have managed to make the worm crawl forward, you can also run the learned policy. For that, we previously wrote the function `voer_beleid_uit()`.
**Assignment**: Use the code cell below to execute and visualize the policy that the worm has learned. Do that for 30 steps. Has the worm

In [ ]:
bestandsnaam = "animatie_worm_na_leerproces.mp4"
# Roep hier de functie voer_beleid_uit op.

Video(bestandsnaam)

# Partners
This educational material is part of the wAIsda? project of Dwengo and is funded with support from VLAIO.
![](img/vlaio.png)![](img/dwengo-groen-zwart.png)